In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
df_pb = pd.read_csv("../benchmark/pacbio/output/qc_summary.tsv", sep='\t')
df_pb[['Donor', 'Tissue', 'Seq_Tech', 'Center', 'Group']] = df_pb['Sample'].str.split('-', expand=True)
#df_pb = df_pb.sort_values(['Donor', 'Tissue', 'Center'])
df_pb['Age'] = np.where(df_pb['Donor'].isin(['ST001', 'ST003']), 'Young', 'Old')
df_pb

In [ ]:

df_ont = pd.read_csv("../benchmark/ont/output/qc_summary.tsv", sep='\t')
df_ont[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = df_ont['Sample'].str.split('-', expand=True)
#df_ont = df_ont.sort_values(['Donor', 'Tissue', 'Center'])
df_ont['Age'] = np.where(df_ont['Donor'].isin(['ST001', 'ST003']), 'Young', 'Old')
df_ont

In [ ]:
comb_meth_df = pd.DataFrame()

for sample in df_pb['Sample']:
    file_path = f'../benchmark/pacbio/output/{sample}/methylation/{sample}.MT.filtered.minimod.mCG.tsv'
    meth_df = pd.read_csv(file_path, sep='\t')
    meth_df['Sample'] = sample
    comb_meth_df = pd.concat([comb_meth_df, meth_df])

for sample in df_ont['Sample']:
    file_path = f'../benchmark/ont/output/{sample}/methylation/{sample}.MT.filtered.minimod.mCG.tsv'
    meth_df = pd.read_csv(file_path, sep='\t')
    meth_df['Sample'] = sample
    comb_meth_df = pd.concat([comb_meth_df, meth_df])

for sample in df_ont['Sample']:
    file_path = f'../benchmark/ont/output/{sample}/methylation/{sample}.MT.filtered.minimod.hCG.tsv'
    meth_df = pd.read_csv(file_path, sep='\t')
    meth_df['Sample'] = sample
    comb_meth_df = pd.concat([comb_meth_df, meth_df])


# shift the '-' strand positions by -1 to aggregate across strands // set strand to '.' // recompute meth freq
comb_meth_df.loc[comb_meth_df["strand"] == "-", "start"] -= 1
comb_meth_df.loc[comb_meth_df["strand"] == "-", "end"] -= 1
comb_meth_df = comb_meth_df.groupby(["contig", "start", "end", "mod_code", "Sample"], as_index=False).agg({"n_called": "sum", "n_mod": "sum"})
comb_meth_df['strand'] = '.'
comb_meth_df["freq"] = comb_meth_df["n_mod"] / comb_meth_df["n_called"]

comb_meth_df = pd.merge(comb_meth_df, df_pb[['Sample', 'Age']], on = 'Sample', how='left')
comb_meth_df = pd.merge(comb_meth_df, df_ont[['Sample', 'Age']], on = 'Sample', how='left')
comb_meth_df['Age'] = comb_meth_df['Age_x'].fillna('') + comb_meth_df['Age_y'].fillna('')

comb_meth_df[['Donor', 'Tissue', 'Seq_Tech', 'Center', 'Group']] = comb_meth_df['Sample'].str.split('-', expand=True)
comb_meth_df['Donor_Tissue'] = comb_meth_df['Donor'] + "_" + comb_meth_df['Tissue']
comb_meth_df = comb_meth_df.drop(['Age_x', 'Age_y'], axis=1)

comb_meth_df['perc'] = comb_meth_df['freq'] * 100
#comb_meth_df = comb_meth_df[~((comb_meth_df['Center'] == 'uwsc') & (comb_meth_df['Seq_Tech'] == 'pacbio'))]

comb_meth_df = comb_meth_df[comb_meth_df['n_called'] > 200]

comb_meth_df_mandh = comb_meth_df.copy()
comb_meth_df_mandh = comb_meth_df_mandh[comb_meth_df_mandh['Seq_Tech'] == 'ont']

comb_meth_df = comb_meth_df[comb_meth_df['mod_code'] == 'm']
comb_meth_df

In [ ]:
# smoothing_window=20

# comb_meth_df['rolling_mean'] = comb_meth_df.groupby(['Sample'])['freq'].transform(
#     lambda x: x.rolling(window=smoothing_window, min_periods=1).mean())

# comb_meth_df

In [ ]:
mean_meth = comb_meth_df.groupby(['Sample', 'Donor', 'Tissue', 'Seq_Tech', 'Center', 'mod_code'])['perc'].mean().reset_index()

mean_meth['Donor_Tissue'] = mean_meth['Donor'] + "_" + mean_meth['Tissue']
mean_meth

In [ ]:
sns.set_theme(style="ticks", context="talk", font_scale=0.75)

g = sns.catplot(
    data=mean_meth[mean_meth['Seq_Tech'] == 'pacbio'],
    x="Donor",
    y="perc",
    col="Tissue",
    #col='Seq_Tech',    
    hue="Center",
    kind="bar",
    height=4,
    aspect=0.8,
    palette="muted",
    legend_out=False,
    sharey=True,
    sharex=False,
    
)

sns.move_legend(g, "center right", bbox_to_anchor=(1.1,0.5))
g.set_titles("{col_name}")
g.set_axis_labels("", "Methylation (%)")
plt.tight_layout()
plt.show()

g = sns.catplot(
    data=mean_meth[(mean_meth['Seq_Tech'] == 'ont')],
    x="Donor",
    y="perc",
    col="Tissue",
    #col='Seq_Tech',    
    hue="Center",
    kind="bar",
    height=4,
    aspect=0.8,
    palette="muted",
    legend_out=False,
    sharey=True,
    sharex=False,
    
)

sns.move_legend(g, "center right", bbox_to_anchor=(1.1,0.5))
g.set_titles("{col_name}")
g.set_axis_labels("", "Methylation (%)")
plt.tight_layout()
plt.show()


g = sns.boxplot(
    data=mean_meth,
    x="Donor_Tissue",
    y="perc",
    hue="Seq_Tech",
 #   bins=6,
 #   alpha=0.6,
)
#plt.ylim(0,20)
plt.yscale('log')
sns.move_legend(g, "center right", bbox_to_anchor=(1.3,0.5))
plt.ylabel("Methyltion (%)")
plt.xticks(rotation=45)
plt.show()

sns.set_theme(style="ticks", context="talk", font_scale=0.75)
g = sns.catplot(
    data=mean_meth,
    x="Donor_Tissue",
    y="perc",
    col="Seq_Tech",
    #col='Seq_Tech',    
    hue="Tissue",
    kind="box",
    height=5,
    aspect=1,
    palette="muted",
    legend_out=False,
    sharey=False,
    sharex=False,
    
)

sns.move_legend(g, "center right", bbox_to_anchor=(1.2,0.5))
g.set_titles("{col_name}")
g.set_axis_labels("", "Methylation (%)")
g.set_xticklabels(rotation=45, horizontalalignment="right")
plt.tight_layout()
plt.show()


In [ ]:
comb_meth_df['index'] = comb_meth_df.groupby(['start']).ngroup()
comb_meth_df['Donor_Tissue'] = comb_meth_df['Donor'] + "_" + comb_meth_df['Tissue']

sns.set_theme(style="ticks", context="talk", font_scale=0.8)

g = sns.relplot(
    data=comb_meth_df[comb_meth_df['Seq_Tech'] == 'ont'],
    x='index',
    y='perc',
    hue='Center',
    row='Donor_Tissue',
    kind='line',
    linewidth=1,
    height=2.5,
    aspect=6
)
sns.move_legend(g, "center right", bbox_to_anchor=(1.15,0.5))
g.set_titles("{row_name}")
g.set_axis_labels("CpG Sites", "Methylation (%)")
plt.tight_layout()
plt.show()

g = sns.relplot(
    data=comb_meth_df[(comb_meth_df['Seq_Tech'] == 'pacbio') & (comb_meth_df['Center'] != 'uwsc')],
    x='index',
    y='perc',
    hue='Center',
    row='Donor_Tissue',
    kind='line',
    linewidth=1,
    height=2.5,
    aspect=6
)
sns.move_legend(g, "center right", bbox_to_anchor=(1.15,0.5))
g.set_titles("{row_name}")
g.set_axis_labels("CpG Sites", "Methylation (%)")
plt.tight_layout()
plt.show()

In [ ]:
sns.set_theme(style="ticks", context="talk", font_scale=0.8)

mean_across_centers = (
    comb_meth_df
    .groupby(['start', 'Seq_Tech'], as_index=False)
    ['perc']
    .mean()
)
mean_across_centers['index'] = mean_across_centers.groupby(['start']).ngroup()

mean_across_centers_pb = (
    comb_meth_df[comb_meth_df['Seq_Tech'] == 'pacbio']
    .groupby(['start', 'Donor', 'Tissue'], as_index=False)
    ['perc']
    .mean()
)
mean_across_centers_pb['index'] = mean_across_centers_pb.groupby(['start']).ngroup()

mean_across_centers_ont = (
    comb_meth_df[comb_meth_df['Seq_Tech'] == 'ont']
    .groupby(['start', 'Donor', 'Tissue'], as_index=False)
    ['perc']
    .mean()
)
mean_across_centers_ont['index'] = mean_across_centers_ont.groupby(['start']).ngroup()


plt.figure(figsize=(20, 3))
sns.lineplot(
    data=mean_across_centers_pb,
    x='index',
    y='perc',
 #   hue='Seq_Tech',
    linewidth=1
)
plt.xlabel("CpG Sites")
plt.ylabel("Methylation (%)")
plt.title("")
#plt.ylim(0, 1)
plt.tight_layout()
#plt.yscale('log')
plt.show()

plt.figure(figsize=(20, 3))
sns.lineplot(
    data=mean_across_centers_ont,
    x='index',
    y='perc',
 #   hue='Seq_Tech',
    color=sns.color_palette()[1],
    linewidth=1
)
plt.xlabel("CpG Sites")
plt.ylabel("Methylation (%)")
plt.title("")
plt.tight_layout()
plt.show()


g = sns.relplot(
    data=mean_across_centers_pb,
    x='index',
    y='perc',
    hue='Donor',
    row='Tissue',
    kind='line',
    linewidth=1.5,
    height=2.5,
    aspect=6
)
sns.move_legend(g, "center right", bbox_to_anchor=(1.1,0.5))
g.set_titles("{row_name}")
g.set_axis_labels("CpG Sites", "Methylation (%)")
plt.tight_layout()
plt.show()

g = sns.relplot(
    data=mean_across_centers_ont,
    x='index',
    y='perc',
    hue='Donor',
    row='Tissue',
    kind='line',
    linewidth=1.5,
    height=2.5,
    aspect=6
)
sns.move_legend(g, "center right", bbox_to_anchor=(1.1,0.5))
g.set_titles("{row_name}")
g.set_axis_labels("CpG Sites", "Methylation (%)")
plt.tight_layout()
plt.show()


In [ ]:
# Pivot table to align ONT and PacBio at the same CpG sites
pivot_df = comb_meth_df[comb_meth_df['Center'] != 'uwsc'].pivot_table(
    index=['start', 'Donor', 'Tissue'],
    columns='Seq_Tech',
    values='perc'
).reset_index()

corr = pivot_df['ont'].corr(pivot_df['pacbio'])

sns.set_theme(style="ticks", context="talk", font_scale=0.8)
sns.lmplot(
    data=pivot_df,
    x='ont',
    y='pacbio',
  #  hue='Tissue',
    scatter_kws={'alpha':0.5, 's':10},
    line_kws={'color':'black'},
    height=5,
    aspect=1.2
)
plt.xlabel("ONT Methylation (%)")
plt.ylabel("PacBio Methylation (%)")
plt.title(f"r = {corr:.2f}")
plt.show()

In [ ]:
# get top 10 methylation sites per averaged across centers? so per donor/tissue combo
mean_huh = (
    comb_meth_df
    .groupby(["start", "Donor", "Tissue", "Seq_Tech"], as_index=False)
    ['perc']
    .mean()
)

mean_huh

top10_per_sample = mean_huh.groupby(["Donor", "Tissue", "Seq_Tech"], group_keys=False).apply(
    lambda x: x.nlargest(10, "perc")
)

top10_per_sample[top10_per_sample['Seq_Tech'] == 'pacbio']


# Pivot so rows=Sample, columns=Site, values=perc
heatmap_df = top10_per_sample[top10_per_sample['Seq_Tech'] == 'ont'].pivot(index=["Donor", "Tissue", "Seq_Tech"], columns="start", values="perc").fillna(0)

plt.figure(figsize=(12,6))
sns.heatmap(heatmap_df, annot=False, fmt=".1f", cmap="viridis")
plt.ylabel("Sample")
plt.xlabel("Site")
plt.title("Top 10 methylated sites per Sample")
plt.tight_layout()
plt.show()

# Pivot so rows=Sample, columns=Site, values=perc
heatmap_df = top10_per_sample[top10_per_sample['Seq_Tech'] == 'pacbio'].pivot(index=["Donor", "Tissue", "Seq_Tech"], columns="start", values="perc").fillna(0)

plt.figure(figsize=(12,6))
sns.heatmap(heatmap_df, annot=False, fmt=".1f", cmap="viridis")
plt.ylabel("Sample")
plt.xlabel("Site")
plt.title("Top 10 methylated sites per Sample")
plt.tight_layout()
plt.show()



In [ ]:
mean_across_centers_mandh = (
    comb_meth_df_mandh.groupby(['start', 'Donor', 'Tissue', 'mod_code'], as_index=False)['perc'].mean()
)
mean_across_centers_mandh['index'] = mean_across_centers_mandh.groupby(['start']).ngroup()
mean_across_centers_mandh['Donor_Tissue'] = mean_across_centers_mandh['Donor'] + "_" + mean_across_centers_mandh['Tissue']

g = sns.relplot(
    data=mean_across_centers_mandh,
    x='index',
    y='perc',
    hue='mod_code',
    row='Donor_Tissue',
    kind='line',
    linewidth=1.5,
    height=2.5,
    aspect=6
)
sns.move_legend(g, "center right", bbox_to_anchor=(1.1,0.5))
g.set_titles("{row_name}")
g.set_axis_labels("CpG Sites", "Methylation (%)")
plt.tight_layout()
plt.show()


In [ ]:
# mamke hist/kde plot to visualzie the higher percent of h vs m meth call

sns.set_theme(style="ticks", context="talk", font_scale=0.8)

g = sns.catplot(
    data=comb_meth_df_mandh,
    x="Tissue",
    y="perc",
    hue="mod_code",
    fill=True,
    kind="bar"
  #  alpha=0.5,
  #  bins=25,
 #   palette="tab10"
)

sns.move_legend(g, "center right", bbox_to_anchor=(1.15,0.5))
g.set_titles("{col_name}")
g.set_axis_labels("", "Methylation (%)")
g.set_xticklabels(rotation=45, horizontalalignment="right")
plt.tight_layout()
plt.show()
